# Market Price Pipeline
**Project:** Detecting and Ranking Underreacted Corporate Disclosures  
**Timeframe:** 2023-01-01 → 2023-12-31  
**Benchmark:** S&P 500 (^GSPC)  
**Tickers:** 50 companies

---
### Pipeline Steps
1. Download raw price data (Yahoo Finance)
2. Clean data — fill gaps, handle splits
3. Build trading calendar
4. Calculate raw daily returns
5. Calculate benchmark returns
6. Compute abnormal returns
7. Filing timestamp alignment helper
8. Post-event return windows (R_short, R_long)
9. Validate on 5 sample stocks
10. Save outputs

## Setup — Install & Import

In [ ]:
# !pip install yfinance pandas numpy

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

print("✓ Libraries loaded")

✓ Libraries loaded


## Configuration

In [ ]:
# ── Study window ──────────────────────────────────────────────
START_DATE  = "2023-01-01"
END_DATE    = "2023-12-31"
BENCHMARK   = "^GSPC"          # S&P 500

# Pull a few extra days for return and R-long window
FETCH_START = "2022-12-15"
FETCH_END   = "2024-01-05"

# ── 50 tickers ────────────────────────────────────────────────
TICKERS = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "META",
    "NVDA", "ADBE", "CRM",
    "JPM",  "BAC",  "GS",   "MS",   "WFC",  "BLK",
    "JNJ",  "PFE",  "MRK",  "ABBV", "UNH",  "LLY",
    "WMT",  "COST", "KO",   "PEP",  "NKE",  "SBUX",
    "BA",   "GE",   "HON",  "CAT",  "UNP",
    "XOM",  "CVX",  "COP",  "SLB",
    "DIS",  "NFLX", "CMCSA","T",    "VZ",
    "MMM",  "DOW",  "DD",   "FCX",
    "TSLA", "INTC", "ORCL", "QCOM",
    "AXP",  "CVS",
]

COMPANY_NAMES = {
    "AAPL":"Apple Inc.", "MSFT":"Microsoft Corporation",
    "GOOGL":"Alphabet Inc.", "AMZN":"Amazon.com Inc.",
    "META":"Meta Platforms Inc.", "NVDA":"NVIDIA Corporation",
    "ADBE":"Adobe Inc.", "CRM":"Salesforce Inc.",
    "JPM":"JPMorgan Chase & Co.", "BAC":"Bank of America Corporation",
    "GS":"Goldman Sachs Group Inc.", "MS":"Morgan Stanley",
    "WFC":"Wells Fargo & Company", "BLK":"BlackRock Inc.",
    "JNJ":"Johnson & Johnson", "PFE":"Pfizer Inc.",
    "MRK":"Merck & Co.", "ABBV":"AbbVie Inc.",
    "UNH":"UnitedHealth Group Incorporated", "LLY":"Eli Lilly and Company",
    "WMT":"Walmart Inc.", "COST":"Costco Wholesale Corporation",
    "KO":"The Coca-Cola Company", "PEP":"PepsiCo Inc.",
    "NKE":"Nike Inc.", "SBUX":"Starbucks Corporation",
    "BA":"Boeing Company", "GE":"General Electric Company",
    "HON":"Honeywell International Inc.", "CAT":"Caterpillar Inc.",
    "UNP":"Union Pacific Corporation", "XOM":"Exxon Mobil Corporation",
    "CVX":"Chevron Corporation", "COP":"ConocoPhillips",
    "SLB":"Schlumberger Limited", "DIS":"The Walt Disney Company",
    "NFLX":"Netflix Inc.", "CMCSA":"Comcast Corporation",
    "T":"AT&T Inc.", "VZ":"Verizon Communications Inc.",
    "MMM":"3M Company", "DOW":"Dow Inc.",
    "DD":"DuPont de Nemours Inc.", "FCX":"Freeport-McMoRan Inc.",
    "TSLA":"Tesla Inc.", "INTC":"Intel Corporation",
    "ORCL":"Oracle Corporation", "QCOM":"Qualcomm Incorporated",
    "AXP":"American Express Company", "CVS":"CVS Health Corporation",
}

OUTPUT_DIR = "pipeline_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"✓ Config ready — {len(TICKERS)} tickers, {START_DATE} to {END_DATE}")

✓ Config ready — 50 tickers, 2023-01-01 to 2023-12-31


---
## Step 1 — Download Price Data

In [ ]:
all_tickers = TICKERS + [BENCHMARK]

raw = yf.download(
    all_tickers,
    start=FETCH_START,
    end=FETCH_END,
    auto_adjust=True,   # handles splits & dividends automatically
    progress=True,
)

# Extract adjusted close prices
if isinstance(raw.columns, pd.MultiIndex):
    prices_all = raw["Close"]
else:
    prices_all = raw[["Close"]]

benchmark_raw = prices_all[BENCHMARK].rename("benchmark")
prices_raw    = prices_all[TICKERS]

print(f"\n✓ Downloaded: {prices_raw.shape[0]} days × {prices_raw.shape[1]} tickers")
print(f"  Date range: {prices_raw.index[0].date()} → {prices_raw.index[-1].date()}")

[*********************100%***********************]  51 of 51 completed


✓ Downloaded: 264 days × 50 tickers
  Date range: 2022-12-15 → 2024-01-04


In [ ]:
benchmark_raw.head(5)

,benchmark
Date,
2022-12-15,3895.750000
2022-12-16,3852.360107
2022-12-19,3817.659912
2022-12-20,3821.620117
2022-12-21,3878.439941


In [ ]:
prices_raw.head(5)

Ticker,AAPL,MSFT,GOOGL,AMZN,META,NVDA,ADBE,CRM,JPM,BAC,...,MMM,DOW,DD,FCX,TSLA,INTC,ORCL,QCOM,AXP,CVS
Date,,,,,,,,,,,,,,,,,,,,,
2022-12-15,134.345627,242.641632,90.115204,88.449997,115.245621,16.934130,328.709991,128.552643,119.674873,29.200859,...,91.121529,40.954323,26.597578,36.473858,157.669998,26.180141,76.680351,108.269135,143.944290,86.318703
2022-12-16,132.387054,238.432144,89.520111,87.860001,118.500084,16.553528,338.540009,126.414032,118.929771,29.136524,...,90.607758,41.177586,26.636797,36.674629,150.229996,25.958357,76.211098,106.310570,140.188049,84.417229
2022-12-19,130.280807,234.300537,87.715027,84.919998,113.588615,16.236868,328.760010,127.143333,119.638062,29.467411,...,90.935379,40.788960,26.499519,35.718567,149.869995,25.833002,77.092148,104.528374,138.951950,84.063461
2022-12-20,130.211899,235.616028,88.290276,85.190002,116.178291,16.068045,338.220001,126.591446,120.217606,29.586897,...,89.952530,41.061821,26.479906,36.139229,137.800003,25.495504,77.513519,104.054962,139.613098,84.019249
2022-12-21,133.312180,238.178787,88.845695,86.769997,118.827499,16.483604,341.380005,128.414658,121.569809,30.037271,...,91.925652,41.896950,26.797615,36.760674,137.570007,25.871572,78.049812,106.366249,140.954636,83.240959


In [ ]:
# Check for missing data
missing = prices_raw.isna().sum()
print("Missing rows per ticker:")
display(missing[missing > 0] if missing.any() else pd.Series(dtype=int, name="(none)"))

Missing rows per ticker:


,(none)


---
## Step 2 — Clean Data

In [ ]:
# Forward-fill short gaps (max 2 consecutive days — handles trading halts)
prices_clean    = prices_raw.ffill(limit=2)
benchmark_clean = benchmark_raw.ffill(limit=2)

# Drop tickers still missing >5% of days after filling
threshold   = 0.05 * len(prices_clean)
still_miss  = prices_clean.isna().sum()
bad_tickers = still_miss[still_miss > threshold].index.tolist()

if bad_tickers:
    print(f"⚠ Dropping tickers with >5% missing: {bad_tickers}")
    prices_clean = prices_clean.drop(columns=bad_tickers)
else:
    print("✓ No tickers exceeded the 5% missing threshold")

# Align to days where benchmark exists
valid_idx       = benchmark_clean.dropna().index
prices_clean    = prices_clean.loc[valid_idx]
benchmark_clean = benchmark_clean.loc[valid_idx]

print(f"✓ Clean shape: {prices_clean.shape[0]} days × {prices_clean.shape[1]} tickers")
prices_clean.head(5)

✓ No tickers exceeded the 5% missing threshold
✓ Clean shape: 264 days × 50 tickers


Ticker,AAPL,MSFT,GOOGL,AMZN,META,NVDA,ADBE,CRM,JPM,BAC,...,MMM,DOW,DD,FCX,TSLA,INTC,ORCL,QCOM,AXP,CVS
Date,,,,,,,,,,,,,,,,,,,,,
2022-12-15,134.345627,242.641632,90.115204,88.449997,115.245621,16.934130,328.709991,128.552643,119.674873,29.200859,...,91.121529,40.954323,26.597578,36.473858,157.669998,26.180141,76.680351,108.269135,143.944290,86.318703
2022-12-16,132.387054,238.432144,89.520111,87.860001,118.500084,16.553528,338.540009,126.414032,118.929771,29.136524,...,90.607758,41.177586,26.636797,36.674629,150.229996,25.958357,76.211098,106.310570,140.188049,84.417229
2022-12-19,130.280807,234.300537,87.715027,84.919998,113.588615,16.236868,328.760010,127.143333,119.638062,29.467411,...,90.935379,40.788960,26.499519,35.718567,149.869995,25.833002,77.092148,104.528374,138.951950,84.063461
2022-12-20,130.211899,235.616028,88.290276,85.190002,116.178291,16.068045,338.220001,126.591446,120.217606,29.586897,...,89.952530,41.061821,26.479906,36.139229,137.800003,25.495504,77.513519,104.054962,139.613098,84.019249
2022-12-21,133.312180,238.178787,88.845695,86.769997,118.827499,16.483604,341.380005,128.414658,121.569809,30.037271,...,91.925652,41.896950,26.797615,36.760674,137.570007,25.871572,78.049812,106.366249,140.954636,83.240959


---
## Step 3 — Build Trading Calendar

In [ ]:
# The calendar = every date we have actual market data within the study window.
# Weekends, holidays, and early-close days are automatically excluded.
calendar = prices_clean.loc[START_DATE:END_DATE].index  # extract the left-most date column

print(f"✓ {len(calendar)} trading days between {START_DATE} and {END_DATE}")
print(f"  First: {calendar[0].date()}   Last: {calendar[-1].date()}")

# Save
pd.DataFrame({"trading_day": calendar}).to_csv(
    f"{OUTPUT_DIR}/06_trading_calendar.csv", index=False
)
print(f"  Saved → {OUTPUT_DIR}/06_trading_calendar.csv")

✓ 250 trading days between 2023-01-01 and 2023-12-31
  First: 2023-01-03   Last: 2023-12-29
  Saved → pipeline_outputs/06_trading_calendar.csv


---
## Step 4 — Calculate Raw Daily Returns

In [ ]:
# Simple percentage return: (P_t / P_{t-1}) - 1
stock_returns_all     = prices_clean.pct_change()
benchmark_returns_all = benchmark_clean.pct_change()

# Restrict to study window
stock_returns     = stock_returns_all.loc[calendar]
benchmark_returns = benchmark_returns_all.loc[calendar]

print(f"✓ Stock returns:     {stock_returns.shape}")
print(f"  Benchmark returns: {benchmark_returns.shape[0]} rows")
stock_returns.head(5)

✓ Stock returns:     (250, 50)
  Benchmark returns: 250 rows


Ticker,AAPL,MSFT,GOOGL,AMZN,META,NVDA,ADBE,CRM,JPM,BAC,...,MMM,DOW,DD,FCX,TSLA,INTC,ORCL,QCOM,AXP,CVS
Date,,,,,,,,,,,,,,,,,,,,,
2023-01-03,-0.037405,-0.001001,0.010087,0.021667,0.036563,-0.020460,0.001159,0.016517,0.007606,0.011775,...,0.021264,0.012899,0.003351,-0.002105,-0.122422,0.011351,0.024223,-0.024923,-0.004264,-0.003005
2023-01-04,0.010314,-0.043743,-0.011670,-0.007924,0.021084,0.030318,0.013327,0.035688,0.009325,0.018800,...,0.021883,0.025274,0.024252,0.028481,0.051249,0.035541,0.009078,0.040392,0.023246,-0.010010
2023-01-05,-0.010604,-0.029638,-0.021344,-0.023726,-0.003376,-0.032816,-0.037990,-0.023282,-0.000222,-0.002050,...,-0.017499,0.011084,0.004821,0.021539,-0.029039,-0.004335,-0.002013,-0.019098,-0.023930,-0.016634
2023-01-06,0.036794,0.011785,0.013225,0.035611,0.024263,0.041640,0.013123,0.030585,0.019136,0.009980,...,0.030579,0.039879,0.022577,0.061245,0.024651,0.042453,0.016012,0.054296,0.025541,0.012714
2023-01-09,0.004089,0.009736,0.007786,0.014870,-0.004230,0.051753,0.027739,0.046901,-0.004132,-0.015112,...,0.000552,0.004725,0.013937,0.013008,0.059349,0.020188,0.012655,-0.006329,0.001532,-0.001201


---
## Step 5 — Benchmark (Market) Returns

In [ ]:
print("S&P 500 daily returns (first 5 trading days of 2023):")
display(benchmark_returns.head(5).rename("S&P 500 return").to_frame())

S&P 500 daily returns (first 5 trading days of 2023):


,S&P 500 return
Date,
2023-01-03,-0.004001
2023-01-04,0.007539
2023-01-05,-0.011646
2023-01-06,0.022841
2023-01-09,-0.000768


---
## Step 6 — Compute Abnormal Returns

Remove the component of each stock's return explained by overall market movement on that day.

In [ ]:
# Broadcast benchmark across all ticker columns
abnormal_returns = stock_returns.sub(benchmark_returns, axis=0)

print(f"✓ Abnormal returns shape: {abnormal_returns.shape}")
abnormal_returns.head(5)

✓ Abnormal returns shape: (250, 50)


Ticker,AAPL,MSFT,GOOGL,AMZN,META,NVDA,ADBE,CRM,JPM,BAC,...,MMM,DOW,DD,FCX,TSLA,INTC,ORCL,QCOM,AXP,CVS
Date,,,,,,,,,,,,,,,,,,,,,
2023-01-03,-0.033404,0.003000,0.014088,0.025667,0.040564,-0.016459,0.005159,0.020517,0.011607,0.015776,...,0.025265,0.016900,0.007352,0.001895,-0.118422,0.015351,0.028224,-0.020922,-0.000264,0.000996
2023-01-04,0.002775,-0.051282,-0.019209,-0.015463,0.013545,0.022779,0.005788,0.028149,0.001786,0.011261,...,0.014344,0.017735,0.016713,0.020942,0.043710,0.028002,0.001539,0.032853,0.015707,-0.017549
2023-01-05,0.001041,-0.017992,-0.009699,-0.012080,0.008269,-0.021170,-0.026344,-0.011637,0.011424,0.009595,...,-0.005854,0.022729,0.016466,0.033184,-0.017394,0.007310,0.009633,-0.007452,-0.012284,-0.004988
2023-01-06,0.013953,-0.011055,-0.009616,0.012770,0.001423,0.018800,-0.009718,0.007744,-0.003705,-0.012861,...,0.007738,0.017038,-0.000264,0.038404,0.001810,0.019612,-0.006828,0.031455,0.002700,-0.010126
2023-01-09,0.004857,0.010504,0.008553,0.015638,-0.003463,0.052521,0.028506,0.047668,-0.003365,-0.014344,...,0.001320,0.005493,0.014704,0.013776,0.060117,0.020956,0.013423,-0.005561,0.002299,-0.000433


---
## Step 7 — Filing Timestamp Alignment Helper

**Rule:**  
- Filed **before 4 PM ET** on a trading day → Day 0 = **same day**  
- Filed **after 4 PM ET** or on a non-trading day → Day 0 = **next trading day**

In [ ]:
def get_event_day0(filing_date: str,
                   filing_time: str,
                   calendar: pd.DatetimeIndex):  # from Step 3

    MARKET_CLOSE = 16  # 4 PM ET

    dt        = pd.Timestamp(f"{filing_date} {filing_time}")
    date      = dt.date()
    hour      = dt.hour
    cal_dates = calendar.normalize()  # makes sure every date is stripped to midnight exactly - only the date part matters

    # The date is actually a trading day & happened before close → Day 0 = same day
    if pd.Timestamp(date) in cal_dates and hour < MARKET_CLOSE:
        return pd.Timestamp(date)

    # Otherwise → next available trading day
    future = cal_dates[cal_dates > pd.Timestamp(date)]
    return future[0] if len(future) > 0 else None


# ── Demo ──────────────────────────────────────────────────────
demo_cases = [
    ("2023-05-04", "14:00", "Before close → same day"),
    ("2023-05-04", "17:30", "After close  → next trading day"),
    ("2023-12-29", "09:00", "Friday AM    → same day"),
    ("2023-12-29", "20:00", "Friday PM    → next Monday"),
]

rows = []
for date, time, note in demo_cases:
    d0 = get_event_day0(date, time, calendar)
    rows.append({"Note": note, "Filing date": date, "Time": time,
                 "Day 0": d0.date() if d0 else "N/A"})

pd.DataFrame(rows)

,Note,Filing date,Time,Day 0
0,Before close → same day,2023-05-04,14:00,2023-05-04
1,After close → next trading day,2023-05-04,17:30,2023-05-05
2,Friday AM → same day,2023-12-29,09:00,2023-12-29
3,Friday PM → next Monday,2023-12-29,20:00,N/A


---
## Step 8 — Post-Event Return Windows
*How did the stock move in the days after this filing?
Did the market react immediately? Or did it drift slowly over the following weeks?*
<br>
| Window | Days | Purpose |
|---|---|---|
| **R_short** | 0 – 1 | Immediate market reaction |
| R_medium | 0 – 3 | Optional diagnostic |
| **R_long** | 5 – 20 | Delayed drift — core signal |
<br>
### ⛳️ Delayed Reaction Flag - training data for the scoring system
- Three conditions: (abs(r_short) < 0.01) and (abs(r_long) > 0.03) and same_dir  
  - Tiny immediate reaction  
  - Big drift later  
  - Same direction  

- Example:  
Day 0-1:  stock moves +0.3%   ← market shrugs  
Days 5-20: stock drifts +5.2%  ← market slowly realizes it's big news  
Same direction: both positive ✓  
delayed = True ✓

In [ ]:
def compute_event_windows(ticker: str,
                          day0: pd.Timestamp,  # from Step 7
                          abnormal_returns: pd.DataFrame,  # from Step 6
                          calendar: pd.DatetimeIndex,
                          short_threshold: float = 0.01,   # default, tune later
                          long_threshold: float  = 0.03    # default, tune later
                          ) -> dict | None:
    """
    Compute Cumulative Abnormal Returns (CAR) for R_short, R_medium, R_long.

    CAR = simple sum of daily abnormal returns (standard event study method).

    Flags event as delayed_reaction if:
        |R_short| < short_threshold  AND
        |R_long|  > long_threshold   AND
        same direction.

    Parameters
    ----------
    short_threshold : float  Max |R_short| to qualify as "weak" reaction (default 1%)
    long_threshold  : float  Min |R_long|  to qualify as "significant" drift (default 3%)
    """
    cal_dates = calendar.normalize()
    day0_norm = day0.normalize()

    if day0_norm not in cal_dates:  # ensure Day 0 actually exists in trading calendar
        return None

    idx = cal_dates.get_loc(day0_norm)  # find the row number of Day 0 in the calendar

    # Compute cumulative abnormal returns (CAR): sum of abnormal returns over the window
    def _car(start_offset: int, end_offset: int):
        s = idx + start_offset
        e = idx + end_offset + 1
        if e > len(cal_dates):  # check whether the window goes beyond the end of calendar
            return None
        window_dates = cal_dates[s:e]  # grab the actual dates
        if ticker not in abnormal_returns.columns:
            return None
        vals = abnormal_returns.loc[window_dates, ticker].dropna()
        return float(vals.sum())

    r_short  = _car(0, 1)
    r_medium = _car(0, 3)
    r_long   = _car(5, 20)

    # Delayed reaction flag
    delayed = None
    if r_short is not None and r_long is not None:
        same_dir = (r_short * r_long) > 0
        delayed  = (
            (abs(r_short) < short_threshold) and
            (abs(r_long)  > long_threshold)  and
            same_dir
        )

    return {
        "ticker"           : ticker,
        "day0"             : day0_norm.date(),
        "R_short_0_1"      : round(r_short,  5) if r_short  is not None else None,
        "R_medium_0_3"     : round(r_medium, 5) if r_medium is not None else None,
        "R_long_5_20"      : round(r_long,   5) if r_long   is not None else None,
        "delayed_reaction" : delayed,
    }

print("✓ compute_event_windows() defined")

✓ compute_event_windows() defined


In [ ]:
# ── Demo: compute windows for a few sample events ─────────────
sample_events = [
    ("AAPL",  "2023-05-04", "14:00"),
    ("MSFT",  "2023-07-25", "17:00"),
    ("TSLA",  "2023-10-18", "08:30"),
    ("NVDA",  "2023-08-23", "16:30"),
    ("JPM",   "2023-01-13", "07:00"),
]

results = []
for tkr, date, time in sample_events:
    d0  = get_event_day0(date, time, calendar)
    win = compute_event_windows(tkr, d0, abnormal_returns, calendar)
    if win:
        results.append(win)

demo_df = pd.DataFrame(results)
display(demo_df)

,ticker,day0,R_short_0_1,R_medium_0_3,R_long_5_20,delayed_reaction
0,AAPL,2023-05-04,0.02576,0.01951,0.00874,False
1,MSFT,2023-07-26,-0.05193,-0.04739,0.00569,False
2,TSLA,2023-10-18,-0.11887,-0.14107,0.06452,False
3,NVDA,2023-08-24,-0.01654,0.02206,-0.12228,False
4,JPM,2023-01-13,0.00781,-0.01235,0.01323,False


---
## Step 9 — Validate on 5 Sample Stocks

Side-by-side check: `raw return − benchmark = abnormal return`

In [ ]:
SAMPLE_TICKERS = ["AAPL", "MSFT", "TSLA", "JPM", "XOM"]
first5 = calendar[:5]

rows = []
for date in first5:
    mkt = benchmark_returns.loc[date]
    for tkr in SAMPLE_TICKERS:
        raw = stock_returns.loc[date, tkr]
        ar  = abnormal_returns.loc[date, tkr]
        rows.append({
            "date"            : date.date(),
            "ticker"          : tkr,
            "raw_return"      : round(raw, 5),
            "benchmark"       : round(mkt, 5),
            "raw − benchmark" : round(raw - mkt, 5),
            "abnormal_return" : round(ar,  5),
            "match ✓/✗"       : "✓" if abs(ar - (raw - mkt)) < 1e-9 else "✗",
        })

val_df = pd.DataFrame(rows)
display(val_df)

all_match = (val_df["match ✓/✗"] == "✓").all()
print(f"\n{'✓ All calculations verified correctly!' if all_match else '✗ Mismatch detected — check pipeline'}")

,date,ticker,raw_return,benchmark,raw − benchmark,abnormal_return,match ✓/✗
0,2023-01-03,AAPL,-0.03740,-0.00400,-0.03340,-0.03340,✓
1,2023-01-03,MSFT,-0.00100,-0.00400,0.00300,0.00300,✓
2,2023-01-03,TSLA,-0.12242,-0.00400,-0.11842,-0.11842,✓
3,2023-01-03,JPM,0.00761,-0.00400,0.01161,0.01161,✓
4,2023-01-03,XOM,-0.03436,-0.00400,-0.03036,-0.03036,✓
5,2023-01-04,AAPL,0.01031,0.00754,0.00278,0.00278,✓
6,2023-01-04,MSFT,-0.04374,0.00754,-0.05128,-0.05128,✓
7,2023-01-04,TSLA,0.05125,0.00754,0.04371,0.04371,✓
8,2023-01-04,JPM,0.00933,0.00754,0.00179,0.00179,✓
9,2023-01-04,XOM,0.00291,0.00754,-0.00463,-0.00463,✓



✓ All calculations verified correctly!


---
## Step 10 — Save All Outputs

In [ ]:
# Restrict price and benchmark to study window before saving
prices_clean.loc[calendar].to_csv(f"{OUTPUT_DIR}/01_prices_clean.csv")
benchmark_clean.loc[calendar].to_csv(f"{OUTPUT_DIR}/02_benchmark_prices.csv")
stock_returns.to_csv(f"{OUTPUT_DIR}/03_stock_returns.csv")
benchmark_returns.to_csv(f"{OUTPUT_DIR}/04_benchmark_returns.csv")
abnormal_returns.to_csv(f"{OUTPUT_DIR}/05_abnormal_returns.csv")
# trading calendar already saved in Step 3

print("✓ All outputs saved to:", OUTPUT_DIR)
for f in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, f)
    size = os.path.getsize(path) / 1024
    print(f"   {f:45s}  {size:6.1f} KB")

✓ All outputs saved to: pipeline_outputs
   01_prices_clean.csv                             225.1 KB
   02_benchmark_prices.csv                           6.6 KB
   03_stock_returns.csv                            264.3 KB
   04_benchmark_returns.csv                          8.0 KB
   05_abnormal_returns.csv                         266.2 KB
   06_trading_calendar.csv                           2.7 KB


---
## Summary

| Output file | Contents |
|---|---|
| `01_prices_clean.csv` | Adjusted close prices, all 50 tickers |
| `02_benchmark_prices.csv` | S&P 500 adjusted close prices |
| `03_stock_returns.csv` | Daily raw returns per ticker |
| `04_benchmark_returns.csv` | Daily S&P 500 returns |
| `05_abnormal_returns.csv` | Daily abnormal returns (raw − benchmark) |
| `06_trading_calendar.csv` | All 250 trading days in 2023 |

**Next step:** Feed actual 8-K filing timestamps from SEC EDGAR into `get_event_day0()` and `compute_event_windows()` to generate the training labels for Layer 1 of the scoring system.  
**Note:** Standardizes the timezones (ET) before passing them to the pipeline.